In [17]:
import pandas as pd
from sklearn.model_selection import train_test_split

In [18]:
df = pd.read_csv("shop_smart_ecommerce.csv")

In [19]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12330 entries, 0 to 12329
Data columns (total 18 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Administrative           12330 non-null  int64  
 1   Administrative_Duration  12330 non-null  float64
 2   Informational            12330 non-null  int64  
 3   Informational_Duration   12330 non-null  float64
 4   ProductRelated           12330 non-null  int64  
 5   ProductRelated_Duration  12330 non-null  float64
 6   BounceRates              12330 non-null  float64
 7   ExitRates                12330 non-null  float64
 8   PageValues               12330 non-null  float64
 9   SpecialDay               12330 non-null  float64
 10  Month                    12330 non-null  object 
 11  OperatingSystems         12330 non-null  int64  
 12  Browser                  12330 non-null  int64  
 13  Region                   12330 non-null  int64  
 14  TrafficType           

In [20]:
# Encoding

from sklearn.preprocessing import LabelEncoder, OneHotEncoder

le=LabelEncoder()
df["Weekend"]=le.fit_transform(df["Weekend"])
df["Revenue"]=le.fit_transform(df["Revenue"])

cols = ["Month","VisitorType"]
ohe = OneHotEncoder(drop="first", sparse_output=False, handle_unknown="ignore")
one_hot_encoded = ohe.fit_transform(df[cols])
one_hot_encoded_df=pd.DataFrame(one_hot_encoded, columns=ohe.get_feature_names_out(cols), index=df.index)

df = pd.concat([df.drop(columns=cols), one_hot_encoded_df], axis=1)

In [21]:
# input and output split

X = df.drop("Revenue", axis=1)
y = df["Revenue"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [22]:
# Model Training

from sklearn.tree import DecisionTreeClassifier

model = DecisionTreeClassifier()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

In [23]:
from sklearn.metrics import f1_score

print("F1-score: ",f1_score(y_test, y_pred))

F1-score:  0.5679314565483476


# Pre-pruning

In [24]:
max_depths = [2, 3, 4, 5, 6, 7, 8, 9, 10]

for depth in max_depths:
    model = DecisionTreeClassifier(max_depth=depth, random_state=42)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    f1_sco = f1_score(y_test, y_pred)
    print(f"For depth={depth}, F1-score is: {f1_sco}")

For depth=2, F1-score is: 0.5256609642301711
For depth=3, F1-score is: 0.6357615894039735
For depth=4, F1-score is: 0.648
For depth=5, F1-score is: 0.5973645680819912
For depth=6, F1-score is: 0.6302521008403361
For depth=7, F1-score is: 0.6276150627615062
For depth=8, F1-score is: 0.6203059805285118
For depth=9, F1-score is: 0.6021220159151194
For depth=10, F1-score is: 0.6167979002624672


# Post-pruning

In [25]:
path = model.cost_complexity_pruning_path(X_train, y_train)
ccp_alphas = path.ccp_alphas

print(ccp_alphas)

[0.00000000e+00 8.77354812e-06 2.03971520e-05 2.09445684e-05
 3.09964942e-05 3.52117426e-05 3.62375110e-05 3.69029607e-05
 4.83317301e-05 5.63215283e-05 5.97874685e-05 6.34719137e-05
 6.67684446e-05 7.24133936e-05 7.39325976e-05 8.06412342e-05
 8.61275407e-05 8.87064071e-05 8.96812028e-05 9.01144453e-05
 9.21197393e-05 9.23931717e-05 9.25632074e-05 9.50425791e-05
 9.60178457e-05 9.60430273e-05 9.83066676e-05 9.97436099e-05
 1.00270321e-04 1.00309791e-04 1.01378751e-04 1.01378751e-04
 1.01378751e-04 1.06664852e-04 1.12673337e-04 1.22997611e-04
 1.27446219e-04 1.33236371e-04 1.34800317e-04 1.35171668e-04
 1.35171668e-04 1.35171668e-04 1.35171668e-04 1.35171668e-04
 1.35171668e-04 1.36113631e-04 1.37364702e-04 1.38243751e-04
 1.44183113e-04 1.44183113e-04 1.44826787e-04 1.45078872e-04
 1.49071021e-04 1.50332117e-04 1.52068127e-04 1.52068127e-04
 1.52068127e-04 1.52068127e-04 1.54481906e-04 1.54481906e-04
 1.54481906e-04 1.57700279e-04 1.57937423e-04 1.58431728e-04
 1.59309466e-04 1.604235

In [26]:
# train out model for all alphas
trees = []

for alpha in ccp_alphas:
    model = DecisionTreeClassifier(random_state=42, ccp_alpha=alpha)
    model.fit(X_train, y_train)

    trees.append((model, alpha))

In [27]:
best_f1 = 0
best_alpha = 0

for model, alpha in trees:
    y_pred = model.predict(X_test)

    curr_f1 = f1_score(y_test, y_pred)

    if curr_f1 > best_f1:
        best_f1 = curr_f1
        best_alpha = alpha

In [28]:
best_f1

0.6742502585315409

# Model After Pruning

In [32]:
model = DecisionTreeClassifier(ccp_alpha=best_alpha, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("F1-score: ",f1_score(y_test, y_pred))

F1-score:  0.6742502585315409
